In [ ]:
# Setup & Imports

import os, json, time, math, random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from tqdm import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())

In [ ]:
# Config & Device Setup

@dataclass
class Config:
    data_root: str = "../data"
    labels_csv: str = "../gt_training.csv"
    out_dir:   str = "../runs/alexnet"
    img_size:  int = 224
    rotate_step_deg: int = 18
    batch_size: int = 32
    epochs: int = 50
    lr: float = 1e-4
    momentum: float = 0.9
    weight_decay: float = 1e-4
    num_workers_mps: int = 0
    num_workers_cuda: int = 4
    use_tensorboard: bool = False

CFG = Config()
OUT = Path(CFG.out_dir); OUT.mkdir(parents=True, exist_ok=True)
print("Saving artifacts to:", OUT.resolve())

# Device selection
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(42)

In [ ]:
# Logger

class Logger:
    def __init__(self):
        self.t0 = time.time()
    def _now(self): 
        return time.strftime("%H:%M:%S")
    def log(self, *msg):
        print(f"[{self._now()}]", *msg)

LOG = Logger()
LOG.log("Logger ready.")

In [ ]:
# Load Labels CSV

DATA_ROOT = Path(CFG.data_root)
LABELS_CSV = Path(CFG.labels_csv)

df = pd.read_csv(LABELS_CSV)
id_col, label_col = df.columns[:2]
df = df[[id_col, label_col]].rename(columns={id_col:"id", label_col:"label"})
LOG.log("CSV loaded:", LABELS_CSV)
display(df.head())
LOG.log("Classes in CSV:", sorted(df["label"].astype(str).unique()))

In [ ]:
# ID Matching Helpers

def _stem_only(s: str) -> str:
    return Path(str(s)).stem.strip().lower()

def _pad5(s: str) -> str:
    t = str(s).strip()
    return t.zfill(5) if t.isdigit() else t

def _keys_from_csv_id(csv_id: str):
    raw = str(csv_id).strip()
    stem = _stem_only(raw)
    stem_padded = _pad5(stem)
    name_png = stem_padded + ".png"
    return {
        stem, stem_padded, name_png,
        "train/"+name_png, "val/"+name_png, "test/"+name_png,
    }

def _keys_from_path(p: Path):
    name = p.name.lower()
    stem = p.stem.lower()
    stem_padded = _pad5(stem)
    rel = str(p.relative_to(p.parents[2])).replace("\\","/").lower() if len(p.parts) >= 3 else name
    return {stem, stem_padded, name, rel, "train/"+name, "val/"+name, "test/"+name}

In [ ]:
# Dataset Class

class Hep2Dataset(Dataset):
    def __init__(self, root_dir, labels_df, split, transform=None):
        self.root = Path(root_dir) / split
        self.transform = transform
        self.id_to_label = {}

        for _, row in labels_df.iterrows():
            for k in _keys_from_csv_id(row["id"]):
                if k:
                    self.id_to_label[k.lower()] = row["label"]

        exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
        self.samples, misses = [], []
        img_files = [q for q in self.root.rglob("*") if q.suffix.lower() in exts]

        for p in sorted(img_files):
            matched = False
            for k in _keys_from_path(p):
                if k in self.id_to_label:
                    self.samples.append((p, self.id_to_label[k]))
                    matched = True
                    break
            if not matched:
                misses.append(p)

        classes = sorted(set(lbl for _, lbl in self.samples))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}

        LOG.log(f"[{split}] matched {len(self.samples)} files; missed {len(misses)}")
        if misses[:5]:
            LOG.log("Unmatched (first few):", [m.name for m in misses[:5]])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        y = self.class_to_idx[label]
        return img, y

In [ ]:
# Data Transforms & Loaders

def get_transforms(img_size=224, is_train=True):
    if is_train:
        transform = transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomCrop((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=180),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, 
                                 saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
    else:
        transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
    return transform

tf_train = get_transforms(CFG.img_size, True)
tf_eval  = get_transforms(CFG.img_size, False)

train_ds = Hep2Dataset(DATA_ROOT, df, split="train", transform=tf_train)
val_ds   = Hep2Dataset(DATA_ROOT, df, split="val",   transform=tf_eval)
test_ds  = Hep2Dataset(DATA_ROOT, df, split="test",  transform=tf_eval)

LOG.log("Detected classes:", train_ds.class_to_idx)

from collections import Counter
train_labels = [label for _, label in train_ds.samples]
class_counts = Counter(train_labels)
LOG.log("Class distribution:", dict(class_counts))

NUM_WORKERS = CFG.num_workers_cuda if DEVICE.type == "cuda" else CFG.num_workers_mps
PERSISTENT  = True if (NUM_WORKERS > 0) else False

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=NUM_WORKERS, persistent_workers=PERSISTENT)
val_loader   = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                          num_workers=NUM_WORKERS, persistent_workers=PERSISTENT)
test_loader  = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False,
                          num_workers=NUM_WORKERS, persistent_workers=PERSISTENT)

xb, yb = next(iter(train_loader))
LOG.log("Sanity batch:", xb.shape, "labels sample:", yb[:10].tolist())

In [ ]:
# Model Setup

def build_model(num_classes):
    net = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
    # Freeze feature layers for fine-tuning
    for param in net.features.parameters():
        param.requires_grad = False
    
    in_features = net.classifier[6].in_features
    net.classifier[6] = nn.Linear(in_features, num_classes)
    return net

model = build_model(num_classes=len(train_ds.class_to_idx)).to(DEVICE)
LOG.log("Model ready on", DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
LOG.log(f"Total params: {total_params:,}, Trainable: {trainable_params:,}")

optimizer = torch.optim.AdamW(model.parameters(),
                              lr=CFG.lr, weight_decay=CFG.weight_decay)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
                                                       factor=0.5, patience=8)
                                                       
criterion = nn.CrossEntropyLoss()

# AMP for CUDA only
USE_AMP = (DEVICE.type == "cuda")
if USE_AMP:
    scaler = torch.amp.GradScaler('cuda')
else:
    scaler = None

In [ ]:
# Training Loop

history = []

def accuracy(output, target):
    with torch.no_grad():
        pred = output.argmax(dim=1)
        return (pred == target).float().mean().item()

def run_epoch(loader, train=False):
    model.train(train)
    running_loss, running_acc, n = 0.0, 0.0, 0
    loop = tqdm(loader, leave=False)
    for x, y in loop:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if train:
            optimizer.zero_grad()
            if USE_AMP and scaler is not None:
                with torch.amp.autocast('cuda'):
                    out = model(x)
                    loss = criterion(out, y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(x)
                loss = criterion(out, y)
                loss.backward()
                optimizer.step()
        else:
            with torch.no_grad():
                out = model(x)
                loss = criterion(out, y)

        bs = y.size(0)
        running_loss += loss.item() * bs
        running_acc  += accuracy(out, y) * bs
        n += bs
        loop.set_postfix(loss=running_loss/n, acc=running_acc/n)
    return running_loss/n, running_acc/n

best_val = 0.0
EPOCHS = CFG.epochs

LOG.log("Starting training for", EPOCHS, "epochs…")
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    scheduler.step(va_acc)
    epoch_time = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']

    history.append({
        "epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
        "val_loss": va_loss, "val_acc": va_acc, "lr": current_lr, "sec": epoch_time
    })

    LOG.log(f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train_acc={tr_acc:.4f} val_acc={va_acc:.4f} "
            f"lr={current_lr:.6f} time={int(epoch_time)}s")

    if va_acc > best_val:
        best_val = va_acc
        torch.save({
            "model_state": model.state_dict(),
            "class_to_idx": train_ds.class_to_idx,
            "config": asdict(CFG)
        }, OUT / "best.pth")
        LOG.log("✔ Saved new best checkpoint (val_acc ↑)")

    if current_lr < 1e-7:
        LOG.log("Learning rate too small, stopping early")
        break

pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
LOG.log("Training done. Best val acc:", round(best_val, 4))

In [ ]:
# Test Evaluation

ckpt = torch.load(OUT / "best.pth", map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
model.eval()

all_preds, all_tgts = [], []
test_loss, test_acc, n = 0.0, 0.0, 0

with torch.no_grad():
    for x, y in tqdm(test_loader, leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = criterion(out, y)
        bs = y.size(0)
        test_loss += loss.item() * bs
        test_acc += accuracy(out, y) * bs
        n += bs
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_tgts.extend(y.cpu().tolist())

test_loss /= n; test_acc /= n
LOG.log(f"TEST — loss={test_loss:.4f} acc={test_acc:.4f}")

# Confusion matrix
cm = confusion_matrix(all_tgts, all_preds)
labels = [ckpt["class_to_idx"][k] for k in sorted(ckpt["class_to_idx"], key=ckpt["class_to_idx"].get)]
inv = {v:k for k,v in ckpt["class_to_idx"].items()}
class_names = [inv[i] for i in range(len(inv))]

print("\nClassification report:\n")
print(classification_report(all_tgts, all_preds, target_names=class_names, digits=4))

plt.figure(figsize=(6,5))
plt.imshow(cm, interpolation='nearest')
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

from collections import Counter
import numpy as np

cm = confusion_matrix(all_tgts, all_preds)
class_acc = cm.diagonal() / cm.sum(axis=1)
mca = np.mean(class_acc)
LOG.log(f"Mean Class Accuracy (MCA): {mca:.4f}")

In [ ]:
# Training Curves

hist = pd.read_csv(OUT / "history.csv")
display(hist.tail())

plt.figure(figsize=(6,4))
plt.plot(hist["epoch"], hist["train_acc"], label="train_acc")
plt.plot(hist["epoch"], hist["val_acc"],   label="val_acc")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("Accuracy vs Epoch"); plt.legend(); plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(hist["epoch"], hist["train_loss"], label="train_loss")
plt.plot(hist["epoch"], hist["val_loss"],   label="val_loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Loss vs Epoch"); plt.legend(); plt.tight_layout()
plt.show()